# Reusable Prompts

This notebook introduces **prompt templates** in LangChain — a way to define reusable, parameterized instructions that you can fill in with different values each time.

## Key concepts

- **`PromptTemplate`** – A simple text template with placeholders (like `{role}`, `{question}`). Used when you only need a single string.
- **`ChatPromptTemplate`** – A chat-style template made up of multiple message types (System, Human, AI). Matches how modern LLMs expect conversations to be formatted.
- **`MessagesPlaceholder`** – A slot in the template where you can inject a list of past messages (conversation history).
- **`HumanMessagePromptTemplate`** / **`AIMessagePromptTemplate`** – Template wrappers for Human and AI message turns with placeholders.

## Why use prompt templates?

Instead of hard-coding prompts in your code, templates let you:
- Reuse the same prompt structure with different inputs
- Keep your prompts consistent and easy to update
- Inject dynamic data (user name, role, history) at runtime

In [ ]:
# Install the core LangChain library.
# The -q flag means "quiet" — suppresses the verbose installation output.
!pip install -q langchain

In [ ]:
# Import message types — represent individual turns in a conversation.
from langchain.messages import HumanMessage, SystemMessage

# Import prompt template classes:
# - ChatPromptTemplate: multi-message (chat-style) template
# - PromptTemplate: simple single-string template
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

# Import helper classes for building each message slot in a chat template:
# - AIMessagePromptTemplate: a templated AI (assistant) message
# - HumanMessagePromptTemplate: a templated Human (user) message
# - MessagesPlaceholder: a slot where a list of messages can be injected at runtime
from langchain_core.prompts.chat import AIMessagePromptTemplate, HumanMessagePromptTemplate, MessagesPlaceholder

In [ ]:
# Create a simple reusable prompt template.
# The curly-brace placeholders {role} and {question} will be filled in later
# when you call .format() or chain it with a model.
prompt_template = PromptTemplate.from_template("Given the user's role ({role}), answer this question:\n{question}")

In [ ]:
# Render the template into a final string by supplying concrete values for the placeholders.
# This just produces the formatted text — it does NOT send it to an AI model yet.
prompt_template.format(role="admin", question="How do I reset ny n8n password?")

In [ ]:
# Build a chat-style prompt template with multiple message slots.
# This mirrors how LLMs actually receive conversations: as an ordered list of messages
# from different "roles" (system, user, assistant).
chat_prompt_template = ChatPromptTemplate.from_messages(
    messages=[
        # A fixed system message — sets the AI's personality/role. Never changes.
        SystemMessage("You are a helpful assistant."),

        # A placeholder for injecting past conversation turns at runtime.
        # 'optional=True' means it's fine if no history is provided — the slot will simply be skipped.
        MessagesPlaceholder(variable_name="history", optional=True),

        # A templated human message. The {job_position} and {company} placeholders
        # will be filled in when .format_messages() is called.
        HumanMessagePromptTemplate.from_template("I am working as a {job_position} in \"{company}\". What is the average salary for my position in other companies?"),

        # A templated AI (assistant) response — demonstrates how to hard-code a canned reply
        # inside the template (useful for few-shot prompting or demos).
        AIMessagePromptTemplate.from_template("The average salary for the \"{job_position}\" in other companies is 500$."),

        # A fixed human follow-up message that always appears after the AI reply above.
        HumanMessage("What other career alternatives do I have nowadays?")
    ]
)

In [ ]:
# Render the chat template into a list of messages without any prior history.
# Notice how the MessagesPlaceholder is simply absent from the output (optional=True).
chat_prompt_template.format_messages(job_position="cashier", company="Local SupaMarket")

In [ ]:
# Render the same template but this time include prior conversation history.
# The history list is injected into the MessagesPlaceholder slot defined above.
# This is how you pass context from previous conversations to the model.
chat_prompt_template.format_messages(job_position="cashier", company="Local SupaMarket", history=[HumanMessage("I am from Bulgaria. I am 46-years-old.")])